In [0]:
# File paths for the three ICLR dataset versions and the OpenAlex dataset.

ICLR24v2_PATH = "/Volumes/workspace/default/ijc427/iclr24v2.parquet"
ICLR25v2_PATH = "/Volumes/workspace/default/ijc427/iclr25v2.parquet"
ICLR26v1_PATH = "/Volumes/workspace/default/ijc427/iclr26v1.parquet"
OPENALEX_PATH = "/Volumes/workspace/default/ijc427/openalex.json"

In [0]:
from pyspark.sql import functions as F

# Load all three ICLR parquet files and the OpenAlex JSON into Spark DataFrames.
# df_24 and df_25 are used for validation against the provided sample outputs.
# df_26 is the final dataset used for all reported results.

df_24 = spark.read.parquet(ICLR24v2_PATH)
df_25 = spark.read.parquet(ICLR25v2_PATH)
df_26 = spark.read.parquet(ICLR26v1_PATH)

openalex = spark.read.json(OPENALEX_PATH)

In [0]:
# check the structure
df_26.printSchema()

root
 |-- year: long (nullable = true)
 |-- id: string (nullable = true)
 |-- title: string (nullable = true)
 |-- abstract: string (nullable = true)
 |-- authors: string (nullable = true)
 |-- author_ids: string (nullable = true)
 |-- decision: string (nullable = true)
 |-- scores: array (nullable = true)
 |    |-- element: long (containsNull = true)
 |-- keywords: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- labels: string (nullable = true)



In [0]:
display(df_26.limit(5))

year,id,title,abstract,authors,author_ids,decision,scores,keywords,labels
2017,B1-Hhnslg,Prototypical Networks for Few-shot Learning,"A recent approach to few-shot classification called matching networks has demonstrated the benefits of coupling metric learning with a training procedure that mimics test. This approach relies on a complicated fine-tuning procedure and an attention scheme that forms a distribution over all points in the support set, scaling poorly with its size. We propose a more streamlined approach, prototypical networks, that learns a metric space in which few-shot classification can be performed by computing Euclidean distances to prototype representations of each class, rather than individual points. Our method is competitive with state-of-the-art one-shot classification approaches while being much simpler and more scalable with the size of the support set. We empirically demonstrate the performance of our approach on the Omniglot and mini-ImageNet datasets. We further demonstrate that a similar idea can be used for zero-shot learning, where each class is described by a set of attributes, and achieve state-of-the-art results on the Caltech UCSD bird dataset.","Jake Snell, Kevin Swersky, Richard Zemel",,Reject,"List(6, 4, 5)","List(deep learning, transfer learning)",transfer learning
2017,B1-q5Pqxl,Machine Comprehension Using Match-LSTM and Answer Pointer,"Machine comprehension of text is an important problem in natural language processing. A recently released dataset, the Stanford Question Answering Dataset (SQuAD), offers a large number of real questions and their answers created by humans through crowdsourcing. SQuAD provides a challenging testbed for evaluating machine comprehension algorithms, partly because compared with previous datasets, in SQuAD the answers do not come from a small set of candidate answers and they have variable lengths. We propose an end-to-end neural architecture for the task. The architecture is based on match-LSTM, a model we proposed previously for textual entailment, and Pointer Net, a sequence-to-sequence model proposed by Vinyals et al. (2015) to constrain the output tokens to be from the input sequences. We propose two ways of using Pointer Net for our tasks. Our experiments show that both of our two models substantially outperform the best results obtained by Rajpurkar et al. (2016) using logistic regression and manually crafted features. Besides, our boundary model also achieves the best performance on the MSMARCO dataset (Nguyen et al. 2016).","Shuohang Wang, Jing Jiang",,Accept (Poster),"List(6, 6, 7)","List(natural language processing, deep learning)",language models
2017,B16Jem9xe,Learning in Implicit Generative Models,"Generative adversarial networks (GANs) provide an algorithmic framework for constructing generative models with several appealing properties: they do not require a likelihood function to be specified, only a generating procedure; they provide samples that are sharp and compelling; and they allow us to harness our knowledge of building highly accurate neural network classifiers. Here, we develop our understanding of GANs with the aim of forming a rich view of this growing area of machine learning---to build connections to the diverse set of statistical thinking on this topic, of which much can be gained by a mutual exchange of ideas. We frame GANs within the wider landscape of algorithms for learning in implicit generative models---models that only specify a stochastic procedure with which to generate data---and relate these ideas to modelling problems in related fields, such as econometrics and approximate Bayesian computation. We develop likelihood-free inference methods and highlight hypothesis testing as a principle for learning in implicit generative models, using which we are able to derive the objective function used by GANs, and many other related objectives. The testing viewpoint directs our focus to the general problem of density ratio estimat

In [0]:
# Shared decision classification function used across all questions.
# Raw decision labels are normalised to three categories: accept, reject, withdrawn.
# Withdrawn is checked first to handle labels containing both withdraw and other keywords.
# Any label not matching the acceptance or withdrawal patterns is classified as rejected.

def label_decision(col):
    d = F.trim(F.lower(col))
    return (
        F.when(d.contains("withdraw"), "withdrawn")
         .when(
             d.contains("accept") | d.contains("spotlight") |
             d.contains("oral") | d.contains("poster") |
             d.contains("talk") | d.contains("invite"),
             "accept"
         )
         .otherwise("reject")
    )

def add_labels(df):
    return df.withColumn("decision_label", label_decision(F.col("decision")))

### Question 1

In [0]:
# check for missing or blank decision values across all three datasets
print("null decisions - 24:", df_24.filter(F.col("decision").isNull()).count())
print("null decisions - 25:", df_25.filter(F.col("decision").isNull()).count())
print("null decisions - 26:", df_26.filter(F.col("decision").isNull()).count())

print("blank decisions - 24:", df_24.filter(F.trim(F.col("decision")) == "").count())
print("blank decisions - 25:", df_25.filter(F.trim(F.col("decision")) == "").count())
print("blank decisions - 26:", df_26.filter(F.trim(F.col("decision")) == "").count())

null decisions - 24: 0
null decisions - 25: 0
null decisions - 26: 0
blank decisions - 24: 0
blank decisions - 25: 0
blank decisions - 26: 19673


In [0]:
# 26v1 has many blank decisions - checking what values actually exist
display(
    df_26
    .select("decision")
    .distinct()
    .orderBy("decision")
)

decision
""
Accept (Oral)
Accept (Poster)
Accept (Spotlight)
Accept (Talk)
Accept (oral)
Accept (poster)
Accept (spotlight)
Accept: notable-top-25%
Accept: notable-top-5%


In [0]:
# Q1: Count accepted, rejected, and withdrawn papers.
# Submissions with a blank decision field are excluded as no category can be assigned.

def decision_counts(df):
    return (
        df.filter(F.trim(F.col("decision")) != "")
          .withColumn("decision_label", label_decision(F.col("decision")))
          .groupBy("decision_label")
          .count()
          .orderBy("decision_label")
    )

print("Q1 Results — iclr24v2 (validation)")
display(decision_counts(df_24))

print("Q1 Results — iclr25v2 (validation)")
display(decision_counts(df_25))

print("Q1 Results — iclr26v1 (final)")
display(decision_counts(df_26))

Q1 Results — iclr24v2 (validation)


decision_label,count
accept,7646
reject,12276
withdrawn,4523


Q1 Results — iclr25v2 (validation)


decision_label,count
accept,11349
reject,17290
withdrawn,7474


Q1 Results — iclr26v1 (final)


decision_label,count
accept,11349
reject,17290
withdrawn,7594


In [0]:
# Extension Q1: Score distribution by decision category.
# Mean score per paper is computed using the aggregate higher-order function.
# Summary statistics are grouped by decision outcome to check whether score profiles differ across accepted, rejected, and withdrawn papers.

def score_summary(df, label):
    result = (
        df
        .withColumn("decision_label", label_decision(F.col("decision")))
        .filter(F.size(F.col("scores")) > 0)
        .withColumn(
            "avg_score",
            F.expr("aggregate(scores, 0D, (acc, x) -> acc + cast(x as double)) / size(scores)")
        )
        .groupBy("decision_label")
        .agg(
            F.round(F.min("avg_score"), 2).alias("min"),
            F.round(F.avg("avg_score"), 2).alias("mean"),
            F.round(F.max("avg_score"), 2).alias("max"),
            F.round(F.stddev("avg_score"), 2).alias("std"),
            F.count("*").alias("papers")
        )
        .orderBy("mean", ascending=False)
    )
    print(f"score distribution by decision ({label}):")
    result.show(truncate=False)
    return result

score_summary(df_26, "iclr26v1")

score distribution by decision (iclr26v1):
+--------------+----+----+----+----+------+
|decision_label|min |mean|max |std |papers|
+--------------+----+----+----+----+------+
|accept        |2.33|6.49|10.0|0.72|11349 |
|reject        |1.0 |4.68|8.0 |1.02|17127 |
|withdrawn     |1.0 |4.02|7.5 |0.87|7141  |
+--------------+----+----+----+----+------+



DataFrame[decision_label: string, min: double, mean: double, max: double, std: double, papers: bigint]

### Question 2

In [0]:
# check how many papers are missing score arrays
print("empty scores - 24:", df_24.filter(F.size(F.col("scores")) == 0).count())
print("empty scores - 25:", df_25.filter(F.size(F.col("scores")) == 0).count())
print("empty scores - 26:", df_26.filter(F.size(F.col("scores")) == 0).count())

empty scores - 24: 341
empty scores - 25: 496
empty scores - 26: 20289


In [0]:
df_24_lb = add_labels(df_24)
df_25_lb = add_labels(df_25)
df_26_lb = add_labels(df_26)

In [0]:
# Q2: Count accepted papers where at least one reviewer scored below 5.
# array_min finds the lowest score in each paper's scores array.
# Papers with empty score arrays are excluded.

def check_low_scores(df, tag):
    print(f"accepted papers with a score below 5 ({tag}):")
    df.filter(F.col("decision_label") == "accept") \
      .filter(F.size(F.col("scores")) > 0) \
      .filter(F.array_min(F.col("scores")) < 5) \
      .agg(F.count("*").alias("count")) \
      .show()

check_low_scores(df_24_lb, "24v2")
check_low_scores(df_25_lb, "25v2")
check_low_scores(df_26_lb, "26v1")

accepted papers with a score below 5 (24v2):
+-----+
|count|
+-----+
| 1140|
+-----+

accepted papers with a score below 5 (25v2):
+-----+
|count|
+-----+
| 1596|
+-----+

accepted papers with a score below 5 (26v1):
+-----+
|count|
+-----+
| 1596|
+-----+



In [0]:

# Extension Q2: Reviewer disagreement for accepted papers.
# Score variance is computed per paper to find where reviewers disagreed most.
# Only accepted papers with more than one score are included.

def reviewer_variance(df, label):
    result = (
        df
        .withColumn("decision_label", label_decision(F.col("decision")))
        .filter(F.col("decision_label") == "accept")
        .filter(F.size(F.col("scores")) > 1)
        .withColumn(
            "avg_score",
            F.expr("aggregate(scores, 0D, (acc, x) -> acc + cast(x as double)) / size(scores)")
        )
        .withColumn(
            "variance",
            F.expr("aggregate(scores, 0D, (acc, x) -> acc + (cast(x as double) - avg_score) * (cast(x as double) - avg_score)) / size(scores)")
        )
        .select("id", "year", "scores", "avg_score", "variance")
        .orderBy("variance", ascending=False)
        .limit(10)
    )
    print(f"top 10 accepted papers by reviewer disagreement ({label}):")
    result.show(truncate=False)
    return result

reviewer_variance(df_26, "iclr26v1")

top 10 accepted papers by reviewer disagreement (iclr26v1):
+----------+----+-------------+-----------------+------------------+
|id        |year|scores       |avg_score        |variance          |
+----------+----+-------------+-----------------+------------------+
|58lbAsXCoZ|2025|[8, 6, 10, 1]|6.25             |11.1875           |
|Hk99zCeAb |2018|[1, 8, 8]    |5.666666666666667|10.888888888888891|
|SkgKO0EtvS|2020|[1, 8, 8]    |5.666666666666667|10.888888888888891|
|HygrdpVKvr|2020|[8, 8, 1]    |5.666666666666667|10.888888888888891|
|SJx9ngStPH|2020|[8, 8, 1]    |5.666666666666667|10.888888888888891|
|ryebG04YvB|2020|[1, 8, 8]    |5.666666666666667|10.888888888888891|
|iI7hZSczxE|2024|[1, 8, 8]    |5.666666666666667|10.888888888888891|
|xHmCdSArUC|2024|[8, 8, 1]    |5.666666666666667|10.888888888888891|
|PJwAkg0z7h|2024|[10, 3, 3, 8]|6.0              |9.5               |
|t9U3LW7JVX|2025|[10, 8, 3, 3]|6.0              |9.5               |
+----------+----+-------------+------------

DataFrame[id: string, year: bigint, scores: array<bigint>, avg_score: double, variance: double]

### Question 3

In [0]:

print("null scores - 24:", df_24.filter(F.col("scores").isNull()).count())
print("null scores - 25:", df_25.filter(F.col("scores").isNull()).count())
print("null scores - 26:", df_26.filter(F.col("scores").isNull()).count())

null scores - 24: 0
null scores - 25: 0
null scores - 26: 0


In [0]:
# Q3: Find the 10 papers with the highest mean reviewer score.
# aggregate sums the scores array and divides by its size to compute the mean.
# Results are ordered by mean score descending, then by year and id to handle ties.

def top_papers(df, tag):
    print(f"top 10 papers by mean reviewer score ({tag}):")
    df.filter(F.size(F.col("scores")) > 0) \
      .withColumn(
          "avg_score",
          F.expr("aggregate(scores, 0D, (acc, x) -> acc + x) / size(scores)")
      ) \
      .filter(F.col("avg_score").isNotNull()) \
      .orderBy(F.col("avg_score").desc(), F.col("year").desc(), F.col("id")) \
      .select("id", "avg_score", "year") \
      .limit(10) \
      .show(truncate=False)

top_papers(df_24, "24v2")
top_papers(df_25, "25v2")
top_papers(df_26, "26v1")

top 10 papers by mean reviewer score (24v2):
+-----------+-----------------+----+
|id         |avg_score        |year|
+-----------+-----------------+----+
|Sy8gdB9xx  |9.666666666666666|2017|
|88nT0j5jAn |9.333333333333334|2023|
|aN4Jf6Cx69 |9.0              |2024|
|Uuf2q9TfXGA|9.0              |2023|
|lTt4KjHSsyl|9.0              |2023|
|b-ny3x071E5|9.0              |2022|
|Hk6kPgZA-  |9.0              |2018|
|r1Ue8Hcxg  |9.0              |2017|
|zEn1BhaNYsC|8.8              |2023|
|UH-cmocLJC |8.75             |2021|
+-----------+-----------------+----+

top 10 papers by mean reviewer score (25v2):
+----------+-----------------+----+
|id        |avg_score        |year|
+----------+-----------------+----+
|u1cQYxRI1H|10.0             |2025|
|Sy8gdB9xx |9.666666666666666|2017|
|6Mxhg9PtDE|9.5              |2025|
|88nT0j5jAn|9.333333333333334|2023|
|LyJi5ugyJx|9.2              |2025|
|DJSZGGZYVi|9.0              |2025|
|Ha6RTeWMd0|9.0              |2025|
|WCRQFlji2q|9.0              |2

In [0]:

# Extension Q3: Acceptance rate by mean score range.
# Each paper's mean score is rounded down to an integer band using floor() so a mean score of 5.7 falls into band 5.
# Acceptance rate per band shows at what score threshold papers tend to be accepted.

def predict_accept(df, label="dataset"):
    result = (
        df
        .withColumn("decision_label", label_decision(F.col("decision")))
        .filter(F.size(F.col("scores")) > 0)
        .withColumn(
            "mean_score",
            F.expr("aggregate(scores, 0D, (acc, x) -> acc + cast(x as double)) / size(scores)")
        )
        .withColumn("score_band", F.floor(F.col("mean_score")).cast("int"))
        .groupBy("score_band")
        .agg(
            F.count("*").alias("total_papers"),
            F.sum(F.when(F.col("decision_label") == "accept", 1).otherwise(0)).alias("accepted"),
        )
        .withColumn("acceptance_rate", F.round(F.col("accepted") / F.col("total_papers") * 100, 2))
        .orderBy("score_band")
    )
    print(f"acceptance rate by score range ({label}):")
    result.show(truncate=False)
    return result

predict_accept(df_26, "iclr26v1")

acceptance rate by score range (iclr26v1):
+----------+------------+--------+---------------+
|score_band|total_papers|accepted|acceptance_rate|
+----------+------------+--------+---------------+
|1         |345         |0       |0.0            |
|2         |968         |1       |0.1            |
|3         |4821        |5       |0.1            |
|4         |8911        |103     |1.16           |
|5         |9532        |1824    |19.14          |
|6         |7788        |6220    |79.87          |
|7         |2696        |2643    |98.03          |
|8         |539         |536     |99.44          |
|9         |16          |16      |100.0          |
|10        |1           |1       |100.0          |
+----------+------------+--------+---------------+



DataFrame[score_band: int, total_papers: bigint, accepted: bigint, acceptance_rate: double]

### Question 4

In [0]:
# check for missing keyword data
print("null keywords - 24:", df_24.filter(F.col("keywords").isNull()).count())
print("null keywords - 25:", df_25.filter(F.col("keywords").isNull()).count())
print("null keywords - 26:", df_26.filter(F.col("keywords").isNull()).count())

print("empty keyword arrays - 24:", df_24.filter(F.size(F.col("keywords")) == 0).count())
print("empty keyword arrays - 25:", df_25.filter(F.size(F.col("keywords")) == 0).count())
print("empty keyword arrays - 26:", df_26.filter(F.size(F.col("keywords")) == 0).count())

null keywords - 24: 0
null keywords - 25: 0
null keywords - 26: 0
empty keyword arrays - 24: 2131
empty keyword arrays - 25: 2131
empty keyword arrays - 26: 2131


In [0]:
# keyword counts per year
kw_counts = (
    df_26
    .select("year", F.explode(F.col("keywords")).alias("kw"))
    .filter(F.col("kw").isNotNull() & (F.trim(F.col("kw")) != ""))
    .withColumn("kw", F.lower(F.trim(F.col("kw"))))
    .groupBy("year", "kw")
    .agg(F.count("*").alias("cnt"))
)

display(kw_counts.orderBy("year", F.desc("cnt")).limit(30))

year,kw,cnt
2017,deep learning,311
2017,natural language processing,88
2017,unsupervised learning,87
2017,computer vision,77
2017,applications,70
2017,supervised learning,66
2017,optimization,48
2017,reinforcement learning,47
2017,theory,32
2017,transfer learning,24


In [0]:
# Q4: Find keywords that appeared fewer than 5 times in one year but more than 50 times the following year.
# Keywords are normalised to lowercase and stripped of whitespace.
# A self-join matches year Y with year Y+1 to find consecutive year pairs where the count crossed the required thresholds.

def keyword_surge(df, tag):
    print(f"keyword surge ({tag}):")

    kw_by_year = (
        df.select("year", F.explode(F.col("keywords")).alias("kw"))
        .filter(F.col("kw").isNotNull() & (F.trim(F.col("kw")) != ""))
        .withColumn("kw", F.lower(F.trim(F.col("kw"))))
        .groupBy("year", "kw")
        .agg(F.count("*").alias("cnt"))
    )

    result = (
        kw_by_year.alias("yr1")
        .join(
            kw_by_year.alias("yr2"),
            (F.col("yr1.kw") == F.col("yr2.kw")) &
            (F.col("yr2.year") == F.col("yr1.year") + 1)
        )
        .filter((F.col("yr1.cnt") < 5) & (F.col("yr2.cnt") > 50))
        .select(
            F.col("yr1.kw").alias("keyword"),
            F.col("yr1.cnt").alias("low_count"),
            F.col("yr2.cnt").alias("high_count"),
            F.col("yr1.year").alias("year")
        )
        .orderBy(F.col("high_count").desc(), F.col("keyword").asc(), F.col("year").asc())
        .limit(10)
    )

    return result

display(keyword_surge(df_24, "24v2"))
display(keyword_surge(df_25, "25v2"))
display(keyword_surge(df_26, "26v1"))

keyword surge (24v2):


keyword,low_count,high_count,year
llm,4,76,2023


keyword surge (25v2):


keyword,low_count,high_count,year
llm,4,76,2023


keyword surge (26v1):


keyword,low_count,high_count,year
test-time scaling,3,87,2025
llm,4,76,2023
large reasoning models,1,57,2025
spatial reasoning,3,57,2025


In [0]:
# Extension Q4: Acceptance rate per keyword.
# The keywords array is exploded so each keyword becomes a separate row.
# Only keywords appearing in at least 50 papers are included to avoid misleading rates from rarely used keywords.

def kw_accept_rate(df, label):
    result = (
        df
        .withColumn("decision_label", label_decision(F.col("decision")))
        .withColumn("kw", F.explode(F.col("keywords")))
        .withColumn("kw", F.trim(F.lower(F.col("kw"))))
        .filter(F.col("kw") != "")
        .groupBy("kw")
        .agg(
            F.count("*").alias("total"),
            F.sum(F.when(F.col("decision_label") == "accept", 1).otherwise(0)).alias("accepted")
        )
        .filter(F.col("total") >= 50)
        .withColumn("rate", F.round(F.col("accepted") / F.col("total") * 100, 2))
        .orderBy("rate", ascending=False)
        .limit(15)
    )
    print(f"acceptance rate by keyword ({label}):")
    result.show(truncate=False)
    return result

kw_accept_rate(df_26, "iclr26v1")

acceptance rate by keyword (iclr26v1):
+-----------------------------+-----+--------+-----+
|kw                           |total|accepted|rate |
+-----------------------------+-----+--------+-----+
|reinforcement learning theory|58   |32      |55.17|
|imagenet                     |50   |23      |46.0 |
|implicit regularization      |74   |33      |44.59|
|nerf                         |52   |23      |44.23|
|applications                 |91   |38      |41.76|
|feature learning             |87   |36      |41.38|
|loss landscape               |59   |24      |40.68|
|neural collapse              |52   |21      |40.38|
|theory                       |193  |77      |39.9 |
|recurrent neural networks    |148  |59      |39.86|
|equivariance                 |157  |62      |39.49|
|batch normalization          |56   |22      |39.29|
|deep learning theory         |149  |58      |38.93|
|neural rendering             |54   |21      |38.89|
|speech recognition           |54   |21      |38.89|
+------

DataFrame[kw: string, total: bigint, accepted: bigint, rate: double]

### Question 5

In [0]:
# Q5: Count ICLR papers that also appear in OpenAlex.
# An inner join on the title column keeps only papers found in both datasets.
# Results are grouped by submission year to show coverage per year.

def openalex_matches(df, tag):
    print(f"OpenAlex matches by year ({tag}):")
    df.join(openalex, on="title", how="inner") \
      .groupBy("year").count() \
      .orderBy("year") \
      .show(truncate=False)

openalex_matches(df_24, "24v2")
openalex_matches(df_25, "25v2")
openalex_matches(df_26, "26v1")

OpenAlex matches by year (24v2):
+----+-----+
|year|count|
+----+-----+
|2017|154  |
|2018|307  |
|2019|199  |
|2020|433  |
|2021|423  |
|2022|52   |
|2023|19   |
|2024|1    |
+----+-----+

OpenAlex matches by year (25v2):
+----+-----+
|year|count|
+----+-----+
|2017|154  |
|2018|309  |
|2019|199  |
|2020|433  |
|2021|423  |
|2022|52   |
|2023|19   |
|2024|1    |
|2025|2    |
+----+-----+

OpenAlex matches by year (26v1):
+----+-----+
|year|count|
+----+-----+
|2017|154  |
|2018|309  |
|2019|199  |
|2020|433  |
|2021|423  |
|2022|52   |
|2023|19   |
|2024|1    |
|2025|2    |
|2026|1    |
+----+-----+



In [0]:
# Extension Q5: Citation count trend over submission years.
# Average, maximum, and minimum citation counts are computed per year
# for all matched papers. Earlier years are expected to have higher averages
# because those papers have had more time to accumulate citations.

def citation_trend(df, label):
    result = (
        df
        .join(openalex.select("title", "cited_by_count"), on="title", how="inner")
        .groupBy("year")
        .agg(
            F.round(F.avg("cited_by_count"), 2).alias("avg_citations"),
            F.round(F.max("cited_by_count"), 2).alias("max_citations"),
            F.round(F.min("cited_by_count"), 2).alias("min_citations"),
            F.count("*").alias("matched_papers")
        )
        .orderBy("year")
    )
    print(f"citation stats by year ({label}):")
    result.show(truncate=False)
    return result

citation_trend(df_26, "iclr26v1")

citation stats by year (iclr26v1):
+----+-------------+-------------+-------------+--------------+
|year|avg_citations|max_citations|min_citations|matched_papers|
+----+-------------+-------------+-------------+--------------+
|2017|279.25       |5915         |0            |154           |
|2018|151.17       |3471         |0            |309           |
|2019|58.45        |1864         |0            |199           |
|2020|42.92        |1010         |0            |433           |
|2021|28.74        |1866         |0            |423           |
|2022|22.83        |696          |1            |52            |
|2023|7.63         |65           |1            |19            |
|2024|1.0          |1            |1            |1             |
|2025|1.0          |2            |0            |2             |
|2026|0.0          |0            |0            |1             |
+----+-------------+-------------+-------------+--------------+



DataFrame[year: bigint, avg_citations: double, max_citations: bigint, min_citations: bigint, matched_papers: bigint]

### Question 6

In [0]:

# Q6: Identify papers with mean reviewer score below 5 but in the top 5%
# of citation counts according to OpenAlex's citation_normalized_percentile field.
# distinct() removes any duplicate matches caused by a title appearing
# more than once in OpenAlex.

def underrated_papers(df, label):
    # papers that got low reviewer scores but turned out to be highly cited
    result = (
        df
        .join(
            openalex.select("title", "cited_by_count", "citation_normalized_percentile"),
            on="title",
            how="inner"
        )
        .filter(F.size(F.col("scores")) > 0)
        .withColumn(
            "avg_score",
            F.expr("aggregate(scores, 0D, (acc, x) -> acc + cast(x as double)) / size(scores)")
        )
        .filter(
            (F.col("avg_score") < 5) &
            (F.col("citation_normalized_percentile.value") >= 0.95)
        )
        .select("id", "year")
        .distinct()
        .orderBy("year")
    )
    print(f"Question 6 results ({label}):")
    result.show(truncate=False)
    print(f"Total: {result.count()}")
    return result

underrated_papers(df_24, "iclr24v2")
underrated_papers(df_25, "iclr25v2")
underrated_papers(df_26, "iclr26v1")

Question 6 results (iclr24v2):
+----------+----+
|id        |year|
+----------+----+
|S19eAF9ee |2017|
|H1bM1fZCW |2018|
|BkPrDFgR- |2018|
|HJWGdbbCW |2018|
|HkmaTz-0W |2018|
|Hy-lXyDWG |2018|
|HJeq43AqF7|2019|
|ByexElSYDr|2020|
|B1esx6EYvr|2020|
|Byg5flHFDr|2020|
|HylvlaVtwr|2020|
|H1emfT4twB|2020|
|S1xFm6VKDH|2020|
+----------+----+

Total: 13
Question 6 results (iclr25v2):
+----------+----+
|id        |year|
+----------+----+
|S19eAF9ee |2017|
|H1bM1fZCW |2018|
|HkmaTz-0W |2018|
|HJWGdbbCW |2018|
|Hy-lXyDWG |2018|
|BkPrDFgR- |2018|
|HJeq43AqF7|2019|
|S1xFm6VKDH|2020|
|ByexElSYDr|2020|
|H1emfT4twB|2020|
|B1esx6EYvr|2020|
|HylvlaVtwr|2020|
|Byg5flHFDr|2020|
+----------+----+

Total: 13
Question 6 results (iclr26v1):
+----------+----+
|id        |year|
+----------+----+
|S19eAF9ee |2017|
|BkPrDFgR- |2018|
|Hy-lXyDWG |2018|
|H1bM1fZCW |2018|
|HkmaTz-0W |2018|
|HJWGdbbCW |2018|
|HJeq43AqF7|2019|
|S1xFm6VKDH|2020|
|B1esx6EYvr|2020|
|HylvlaVtwr|2020|
|H1emfT4twB|2020|
|ByexElSYDr|2020|
|By

DataFrame[id: string, year: bigint]

In [0]:
# Extension Q6: Citation impact by decision category.
# Average and median citation counts are computed per decision category.
# Both are included because citation counts are often skewed by a small number of very highly cited papers.

def citations_per_decision(df, label):
    result = (
        df
        .withColumn("decision_label", label_decision(F.col("decision")))
        .join(openalex.select("title", "cited_by_count"), on="title", how="inner")
        .groupBy("decision_label")
        .agg(
            F.round(F.avg("cited_by_count"), 2).alias("avg_citations"),
            F.round(F.max("cited_by_count"), 2).alias("max_citations"),
            F.round(F.median("cited_by_count"), 2).alias("median_citations"),
            F.count("*").alias("papers")
        )
        .orderBy("avg_citations", ascending=False)
    )
    print(f"citation impact by decision category ({label}):")
    result.show(truncate=False)
    return result

citations_per_decision(df_26, "iclr26v1")

citation impact by decision category (iclr26v1):
+--------------+-------------+-------------+----------------+------+
|decision_label|avg_citations|max_citations|median_citations|papers|
+--------------+-------------+-------------+----------------+------+
|reject        |108.25       |5915         |23.0            |236   |
|accept        |79.68        |3471         |22.0            |1320  |
|withdrawn     |72.92        |696          |18.0            |37    |
+--------------+-------------+-------------+----------------+------+



DataFrame[decision_label: string, avg_citations: double, max_citations: bigint, median_citations: double, papers: bigint]